# Token Prediction Dynamics

Track how predictions form and evolve across layers: commitment timing,
competition between alternatives, confidence trajectories, resolution patterns,
and position difficulty.

## Why This Matters

Token prediction dynamics tracks how individual token representations evolve through the network. Understanding token-level dynamics reveals how context is integrated, when predictions form, and how different positions interact to produce the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_prediction_dynamics import (
    per_position_commitment,
    prediction_competition,
    confidence_evolution,
    prediction_resolution_pattern,
    position_difficulty,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Per-Position Commitment

Find the layer where the model first commits to its final prediction at each position.

In [ ]:
result = per_position_commitment(model, tokens)
print(f"Mean commit layer: {result['mean_commit_layer']:.2f}")
print(f"Early commit fraction: {result['early_commit_fraction']:.2%}\n")
for p in result['per_position']:
    status = '(early)' if p['early_commit'] else '(late)'
    print(f"  Pos {p['position']}: commits at layer {p['commit_layer']} {status}, "
          f"final token = {p['final_token']}")

## Prediction Competition

Track competition between the top-k candidate predictions across layers.

In [ ]:
result = prediction_competition(model, tokens, position=-1, top_k=3)
print(f"Tracking tokens: {result['tracked_tokens']}")
print(f"Lead changes: {result['n_lead_changes']}\n")
for stage in result['stages']:
    leader_str = f"leader={stage['leader']} ({stage['leader_prob']:.2%})"
    print(f"  Layer {stage['layer']}: {leader_str}")
    for c in stage['competitors']:
        marker = ' ◄' if c['token'] == stage['leader'] else ''
        print(f"    token {c['token']:3d}: prob={c['prob']:.4f}, rank={c['rank']}{marker}")

## Confidence Evolution

Track how confident the model becomes about its prediction at each position.

In [ ]:
result = confidence_evolution(model, tokens)
print(f"Mean final confidence: {result['mean_final_confidence']:.4f}\n")
for p in result['per_position']:
    trend = '↑' if p['confidence_trend'] > 0 else '↓'
    confs = [f"{t['confidence']:.3f}" for t in p['trajectory']]
    print(f"  Pos {p['position']}: {' → '.join(confs)} {trend} (trend={p['confidence_trend']:+.4f})")

## Prediction Resolution Patterns

Classify how predictions resolve: constant, single switch, gradual, or unstable.

In [ ]:
result = prediction_resolution_pattern(model, tokens)
print('Pattern counts:', result['pattern_counts'])
print()
for p in result['per_position']:
    preds = ' → '.join(str(x) for x in p['predictions'])
    print(f"  Pos {p['position']}: {p['pattern']:14s} ({p['n_changes']} changes) [{preds}]")

## Position Difficulty

Score positions by prediction difficulty: low confidence + many changes = hard.

In [ ]:
result = position_difficulty(model, tokens)
print(f"Hardest position: {result['hardest_position']}")
print(f"Easiest position: {result['easiest_position']}\n")
for p in result['per_position']:
    hard_marker = ' ★' if p['is_hard'] else ''
    print(f"  Pos {p['position']}: difficulty={p['difficulty']:.4f}, "
          f"mean_conf={p['mean_confidence']:.4f}, changes={p['n_changes']}{hard_marker}")